# 04b - Python Intricacies

This notebook covers `numpy`/`pandas` basics, a concrete duck typing demo, and a minimal `scikit-learn` training loop on a toy robotics dataset. Read `concept.md` first for the dynamic/duck typing background and why Python dominates the ML ecosystem.

## Imports

In [1]:
import numpy as np
import pandas as pd

np.random.seed(1515)  # team number, for reproducibility -- see ml_resources for the same convention

## numpy Basics

`numpy` arrays are the foundation nearly everything else in this notebook builds on. Unlike a plain Python list, a numpy array holds a single, fixed data type and supports fast, *vectorized* math — operating on every element at once instead of writing an explicit loop.

In [2]:
# Simulated distances (feet) from ten practice shots
distances = np.array([8.0, 12.5, 6.0, 15.0, 9.5, 20.0, 4.0, 11.0, 7.5, 13.0])

print("mean distance:", distances.mean())
print("max distance: ", distances.max())

# Vectorized operation: convert every distance from feet to meters, no loop needed
distances_m = distances * 0.3048
print("distances in meters:", np.round(distances_m, 2))

mean distance: 10.65
max distance:  20.0
distances in meters: [2.44 3.81 1.83 4.57 2.9  6.1  1.22 3.35 2.29 3.96]


`distances * 0.3048` multiplies every element of the array by `0.3048` in one step. Writing the equivalent with a plain Python list would require a loop or a list comprehension — numpy does it faster and in less code, which is why virtually every numeric library in this ecosystem is built on top of it.

## pandas Basics

`pandas` builds tabular data (rows and columns, like a spreadsheet) on top of numpy. A `DataFrame` is the core structure — here, a simple shot log.

In [3]:
shot_log = pd.DataFrame({
    "distance_ft": [8.0, 12.5, 6.0, 15.0, 9.5, 20.0, 4.0, 11.0, 7.5, 13.0],
    "angle_deg": [5, -10, 2, 20, -5, 25, 0, 15, -3, 18],
    "made": [1, 0, 1, 0, 1, 0, 1, 0, 1, 0],
})

print(shot_log.head())
print()
print(shot_log.describe())

   distance_ft  angle_deg  made
0          8.0          5     1
1         12.5        -10     0
2          6.0          2     1
3         15.0         20     0
4          9.5         -5     1

       distance_ft  angle_deg       made
count    10.000000   10.00000  10.000000
mean     10.650000    6.70000   0.500000
std       4.708444   11.96337   0.527046
min       4.000000  -10.00000   0.000000
25%       7.625000   -2.25000   0.000000
50%      10.250000    3.50000   0.500000
75%      12.875000   17.25000   1.000000
max      20.000000   25.00000   1.000000


In [4]:
# groupby: average distance for made vs. missed shots
print(shot_log.groupby("made")["distance_ft"].mean())

made
0    14.3
1     7.0
Name: distance_ft, dtype: float64


`.describe()` gives a quick statistical summary of every numeric column at once — count, mean, min/max, quartiles. `.groupby("made")` splits the rows into groups by that column's value, so we can compare, for instance, the average shot distance between makes and misses in one line.

## Duck Typing in Practice

`concept.md` describes duck typing abstractly: Python doesn't check an object's declared type before calling a method, only whether the object actually has that method when it's called. Here's a function that works identically on a plain Python list, a numpy array, and a pandas Series — three completely unrelated classes — because all three happen to support the same operations (`sum()` and `len()`).

In [5]:
def average(values):
    """Works on ANYTHING that supports sum() and len() -- no type check anywhere."""
    return sum(values) / len(values)

plain_list = [8.0, 12.5, 6.0]
numpy_array = np.array([8.0, 12.5, 6.0])
pandas_series = shot_log["distance_ft"]

print("list:   ", average(plain_list))
print("array:  ", average(numpy_array))
print("series: ", average(pandas_series))

list:    8.833333333333334
array:   8.833333333333334
series:  10.65


`average()` never once mentions `list`, `numpy.ndarray`, or `pandas.Series` — it doesn't need to. In Java, you'd need all three types to formally implement some shared interface before a single function could accept any of them (see `02_oop_inheritance`). In Python, they simply all happen to support `sum()` and `len()`, and that's enough.

## A Minimal Training Loop with scikit-learn

Now the payoff: using a larger, synthetic version of the shot log above to train an actual classifier that predicts whether a shot is made, based on distance and angle. This is deliberately small and simple — `scikit-learn`'s `LogisticRegression` needs no manual gradient descent code, no GPU, and trains in a fraction of a second, which is exactly why it's a good first model to reach for before anything heavier.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Generate a bigger synthetic dataset: shots are more likely to be made when
# closer to the goal and closer to straight-on (small angle).
n_shots = 500
distance_ft = np.random.uniform(2, 25, n_shots)
angle_deg = np.random.uniform(-30, 30, n_shots)

# A "difficulty" score: higher distance and higher |angle| make a shot harder.
difficulty = 0.15 * distance_ft + 0.08 * np.abs(angle_deg)
make_probability = 1 / (1 + np.exp(difficulty - 3))  # logistic curve
made = (np.random.uniform(0, 1, n_shots) < make_probability).astype(int)

X = np.column_stack([distance_ft, angle_deg])
y = made

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1515)

model = LogisticRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)
print(f"test accuracy: {accuracy_score(y_test, predictions):.2f}")
print(f"learned coefficients (distance, angle): {model.coef_[0]}")

test accuracy: 0.69
learned coefficients (distance, angle): [-0.14515755 -0.00915888]


The coefficients should both come out negative: increasing distance or increasing the angle away from straight-on both make a shot *less* likely to be made, which is exactly the relationship we built into `make_probability` when generating the data. That's the whole training loop: prepare features (`X`) and labels (`y`), split into train/test sets, `fit()` the model on the training set, then check how well it generalizes on data it never saw during training (the test set).

## Try It Yourself

No solutions are provided — these are meant to be worked through on your own or with a mentor.

1. Add a new column to `shot_log` for shot type (e.g. `"layup"` vs. `"three"`) and use `.groupby()` to compare make rates between types.
2. Write a duck-typed function like `average()` that instead computes the *maximum* of any sum/len-free collection (hint: look for a different pair of operations every relevant type supports).
3. Retrain the `LogisticRegression` model with `test_size=0.5` instead of `0.2` and see how the reported accuracy changes. Why might a smaller training set make the model less reliable?

In [7]:
# Your code here


## Resources

- [numpy Quickstart](https://numpy.org/doc/stable/user/quickstart.html) - the official basics of arrays and vectorized operations.
- [10 Minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) - a fast tour of `DataFrame`/`Series` operations.
- [scikit-learn: `LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) - the model used above, and what its hyperparameters control.